# 01 Tongue Overview

This notebook shows the Tongue hindcast handoff artifact.

Full workflow context: [Tongue Data Assembly](../workflows/tongue-data-assembly.md)

Artifacts read here: the hindcast container, the latest health report, and the latest run report.

In [1]:
from pathlib import Path

from swimrs.container import SwimContainer

ROOT = Path("/nas/swim/examples/tongue_ensemble")
HINDCAST_CONTAINER = ROOT / "data" / "tongue_hindcast.swim"
HEALTH_REPORT_DIR = (
    HINDCAST_CONTAINER.parent / f"{HINDCAST_CONTAINER.name}.reports" / "health"
)
RUN_REPORT_DIR = (
    HINDCAST_CONTAINER.parent / f"{HINDCAST_CONTAINER.name}.reports" / "run"
)

container = SwimContainer.open(str(HINDCAST_CONTAINER), mode="r")
HINDCAST_CONTAINER

PosixPath('/nas/swim/examples/tongue_ensemble/data/tongue_hindcast.swim')

In [2]:
print(container.query.status())

CONTAINER STATUS
  URI: file:///mnt/mco_nas1/dgketchum/swim/examples/tongue_ensemble/data/tongue_hindcast.swim
  Storage: directory
  Fields: 1999
  Date range: 1987-01-01 to 2025-12-31
  Days: 14245
  Default restart: hindcast_initial_state

SIMULATION RUNS:
----------------------------------------
  hindcast_core: role=core, profile=core, dates=1987-01-01..2025-12-31, fields=1999
  hindcast_initial_state: role=initialization, profile=state_only, dates=1987-01-01..1991-12-31, fields=1999

DATA PATHS:
----------------------------------------

  calibration/
    metadata/batch_id: shape=(1999,), dtype=int32
    metadata/calibrated: shape=(1999,), dtype=uint8
    parameters/aw: shape=(1999,), 100.0% valid
    parameters/kr_damp: shape=(1999,), 100.0% valid
    parameters/ks_damp: shape=(1999,), 100.0% valid
    parameters/mad: shape=(1999,), 100.0% valid
    parameters/ndvi_0: shape=(1999,), 100.0% valid
    parameters/ndvi_k: shape=(1999,), 100.0% valid
    parameters/swe_alpha: shape=(

In [3]:
(
    sorted(HEALTH_REPORT_DIR.iterdir())[-1],
    sorted(RUN_REPORT_DIR.iterdir())[-1],
    container.list_runs(),
)

(PosixPath('/nas/swim/examples/tongue_ensemble/data/tongue_hindcast.swim.reports/health/20260316T155844136283Z'),
 PosixPath('/nas/swim/examples/tongue_ensemble/data/tongue_hindcast.swim.reports/run/20260312T164259025759Z'),
 ['hindcast_core', 'hindcast_initial_state'])

In [4]:
container.runs.metadata("hindcast_core")

{'run_id': 'hindcast_core',
 'created_at': '2026-03-16T17:37:57.692942+00:00',
 'status': 'completed',
 'profile': 'core',
 'engine': 'python',
 'field_count': 1999,
 'n_days': 14245,
 'start_date': '1987-01-01',
 'end_date': '2025-12-31',
 'refet_type': 'eto',
 'etf_model': 'ensemble',
 'met_source': 'gridmet',
 'mask_mode': 'irrigation',
 'ndvi_mode': 'observed',
 'persisted_outputs': ['eta',
  'etf',
  'runoff',
  'dperc',
  'irr_sim',
  'gw_sim',
  'swe',
  'depl_root'],
 'restart_from': 'hindcast_initial_state',
 'spinup_json_path': None,
 'calibrated_params_path': None,
 'container_uri': 'file:///mnt/mco_nas1/dgketchum/swim/examples/tongue_ensemble/data/tongue_hindcast.swim',
 'command': '/home/dgketchum/code/swim-mtdnrc/.venv/bin/swim run /nas/swim/examples/tongue_ensemble/tongue_ensemble.toml --container /nas/swim/examples/tongue_ensemble/data/tongue_hindcast.swim --run-id hindcast_core --profile core --ndvi-mode observed --overwrite --workers 40'}

In [5]:
ds = container.open_run_dataset(
    "hindcast_core",
    variables=["eta", "dperc", "swe"],
    start_date="2023-05-01",
    end_date="2023-05-15",
)
ds

<xarray.Dataset> Size: 392kB
Dimensions:  (time: 15, site: 1999)
Coordinates:
  * time     (time) datetime64[ns] 120B 2023-05-01 2023-05-02 ... 2023-05-15
  * site     (site) <U4 32kB '1917' '1918' '1919' ... '1914' '1915' '1916'
Data variables:
    eta      (time, site) float32 120kB 3.689 3.298 1.261 ... 3.776 3.68 4.056
    dperc    (time, site) float32 120kB 0.409 0.3657 0.0 ... 0.7051 0.6857 0.0
    swe      (time, site) float32 120kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0